# Construction et Optimisation du Moteur de Prédiction

## Objectif

Cette étape a pour but d'entraîner, d'évaluer et d'optimiser différents algorithmes de Machine Learning afin de sélectionner le modèle le plus performant pour prédire le rendement agricole (`Yield_tons_per_hectare`).

La démarche suit les standards MLOps, incluant un suivi rigoureux des expérimentations, et se déroulera selon les étapes suivantes :

- **Préparation des données** : Séparation du jeu de données en ensembles d'entraînement et de test (*Train/Test split*).
- **Pipeline de pré-traitement** : Standardisation des variables numériques (*StandardScaler*) et encodage des variables catégorielles (*One-Hot Encoding*).
- **Expérimentations et Tracking** : Entraînement de plusieurs modèles de régression (ex: Random Forest, XGBoost) avec journalisation systématique via **MLflow**.
- **Évaluation des performances** : Analyse des modèles à l'aide des métriques de régression (RMSE et R²).
- **Optimisation** : Recherche des meilleurs hyperparamètres (Fine-tuning) pour le modèle retenu.
- **Sérialisation** : Sauvegarde (*persistance*) du modèle final prêt à être déployé en production via l'API.

## Préparation des données

In [2]:
import pandas as pd

#Importation du dataframe
df = pd.read_csv("../data/processed/dataset_ml_ready.csv")

display(df.head(5))


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare,pesticides_tonnes_mean
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816,0.000000
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341,36942.215995
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443,0.000000
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573,40752.554896
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251,35453.212930


In [3]:
from sklearn.model_selection import train_test_split

#Séparation des données
y = df["Yield_tons_per_hectare"]
X = df.drop(columns=["Yield_tons_per_hectare"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train")
display(X_train.head())
print("X_test")
display(X_test.head())
print("y_train")
display(y_train.head())
print("y_test")
display(y_test.head())



X_train


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,pesticides_tonnes_mean
566853,North,Sandy,Soybean,887.383906,33.050569,True,True,Cloudy,79,40752.554896
382311,South,Clay,Soybean,179.188444,24.699056,False,False,Rainy,88,40752.554896
241519,South,Silt,Wheat,649.310754,30.057577,False,False,Cloudy,120,35453.212930
719220,North,Loam,Cotton,221.489100,19.508065,False,True,Cloudy,101,0.000000
905718,East,Peaty,Rice,737.016449,18.233438,True,False,Rainy,88,36942.215995


X_test


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,pesticides_tonnes_mean
987231,West,Silt,Cotton,714.854403,23.875872,False,False,Sunny,120,0.000000
79954,North,Chalky,Cotton,860.604672,23.070897,False,False,Rainy,78,0.000000
567130,North,Sandy,Barley,802.081954,24.020125,True,True,Rainy,140,0.000000
500891,West,Chalky,Cotton,203.616909,16.895211,False,True,Sunny,96,0.000000
55399,East,Silt,Rice,510.528102,18.402903,False,True,Cloudy,65,36942.215995


y_train


566853    8.285534
382311    0.760165
241519    4.563030
719220    2.872347
905718    4.809969
Name: Yield_tons_per_hectare, dtype: float64

y_test


987231    3.840988
79954     5.138173
567130    6.401523
500891    2.658805
55399     2.797703
Name: Yield_tons_per_hectare, dtype: float64

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

#Encodage des variables 
numeric_features =[
    "Rainfall_mm",
    "Temperature_Celsius",
    "Days_to_Harvest",
    "pesticides_tonnes_mean",
]

categorical_features = [
    "Region",
    "Soil_Type",
    "Crop",
    "Weather_Condition",
]

boolean_features = ["Fertilizer_Used", "Irrigation_Used"]

categorical_transformer = OneHotEncoder(
    drop="first",
    handle_unknown="ignore"
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
        (
            "bool",
            "passthrough",
            boolean_features
        )
    ]
)